### LCEL - 랭체인 문법
- Model, Prompt, Output Parser로 주로 구성됨
- 각각을 부품처럼 갈아 끼우기 좋게 체인으로 연결해서 사용
- 실행 -> invoke

- Model : LLM은 언제든 바뀔 수 있음 
    - -> openai, gemini, claude, 로컬모델(gemma, qwen)
    - 한국 모델: LG 엑사원(오픈소스), KT 마음, SK 에이닷, NC 바르코 ...

- Prompt : 프롬프트를 템플릿으로 관리(파일로 저장해서 불러와서 관리 가능)
- Parser : 답을 원하는 형식으로 다듬는 부분
- Chain : chain = prompt | model | parser 형태로 연결

- LangChain : 각각 부품화해서 조립하기 쉽게 해 주는 라이브러리.

### Model 파트

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="openai:gpt-5.6-luna",
    reasoning={"effort" : "none"} # 추론 정도 정하기 -> none은 추론 낮음, low, medium, high, max 등 있음
)

# 가장 단순한 모델 호출
result = model.invoke("파이썬에 대해서 알려줘")
print(result.text)

파이썬(Python)은 배우기 쉽고 활용 범위가 넓은 **프로그래밍 언어**입니다. 문법이 간결하고 읽기 쉬워 초보자부터 전문가까지 많이 사용합니다.

## 주요 특징

- **쉬운 문법**: 영어 문장처럼 읽기 쉬움
- **무료·오픈 소스**
- **다양한 운영체제 지원**: Windows, macOS, Linux 등
- **풍부한 라이브러리**: 웹 개발, 데이터 분석, 인공지능 등에 활용
- **인터프리터 언어**: 코드를 작성하고 바로 실행하며 결과 확인 가능

## 간단한 예제

```python
name = "철수"
print(f"안녕하세요, {name}님!")
```

실행 결과:

```text
안녕하세요, 철수님!
```

## 기본 문법

### 변수

```python
age = 20
name = "민수"
```

### 조건문

```python
age = 20

if age >= 18:
    print("성인입니다.")
else:
    print("미성년자입니다.")
```

파이썬은 들여쓰기로 코드 블록을 구분합니다.

### 반복문

```python
for i in range(5):
    print(i)
```

결과:

```text
0
1
2
3
4
```

### 함수

```python
def add(a, b):
    return a + b

result = add(3, 5)
print(result)
```

### 리스트

```python
fruits = ["사과", "바나나", "포도"]

for fruit in fruits:
    print(fruit)
```

## 파이썬의 주요 활용 분야

- **웹 개발**: Django, Flask, FastAPI
- **데이터 분석**: NumPy, pandas
- **인공지능·머신러닝**: PyTorch, TensorFlow, scikit-learn
- **자동화**: 파일 처리, 웹 브라우저 자동화, 반복 업무
- **게임 개발**: Pygame
- **데스크톱 

### Prompt 설정 - 부품이라고 생각

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 번역가야. {language}로 아래 내용을 번역해주세요"),
    ("human", "{question}")
])

# prompt도 invoke 가능하다!
tmp_prompt = prompt.invoke({"language" : "영어", "question" : "나는 금요일이라서 너무 좋다. 주말에 공부를 더 할 수 있어서"})

messages=[SystemMessage(content='당신은 번역가야. 영어로 아래 내용을 번역해주세요', additional_kwargs={}, response_metadata={}), HumanMessage(content='나는 금요일이라서 너무 좋다. 주말에 공부를 더 할 수 있어서', additional_kwargs={}, response_metadata={})]


In [ ]:
chain = prompt | model
result = chain.invoke({"language" : "영어", "question" : "나는 금요일이라서 너무 좋다. 주말에 공부를 더 할 수 있어서"})

content=[{'type': 'text', 'text': 'I’m so happy it’s Friday because I can study more over the weekend.', 'annotations': [], 'id': 'msg_0f67587b16b4e5cb006aaccac8d96887d09851e94e1ba6dee3', 'phase': 'final_answer'}] additional_kwargs={} response_metadata={'id': 'resp_0f67587b16b4e5cb006aaccac822b487d083c7e6a75827d688', 'created_at': 1789709000.0, 'metadata': {}, 'model': 'gpt-5.6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna'} id='resp_0f67587b16b4e5cb006aaccac822b487d083c7e6a75827d688' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 43, 'output_tokens': 20, 'total_tokens': 63, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}}


In [ ]:
print(result.text)

I’m so happy it’s Friday because I can study more over the weekend.


### 출력 형식 - Parser

In [10]:
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser

chain = prompt | model | StrOutputParser()

result = chain.invoke({"language" : "영어", "question" : "주말에 할 일 알려줘. 리스트 형식으로 줘"})
print(result)

Tell me what to do this weekend. Please provide it in a list format.


In [11]:
chain = prompt | model | CommaSeparatedListOutputParser()

result = chain.invoke({"language" : "영어", "question" : "주말에 할 일 알려줘. 리스트 형식으로 줘"})
print(result)

['Tell me what to do this weekend. Please provide it in list format.']


In [ ]:
# parser 대신 프롬프트 바꿔보기

prompt = ChatPromptTemplate.from_messages([
    ("system", "{language}로 아래 내용에 답해주세요"),
    ("human", "{question}")
])

chain = prompt | model 
result = chain.invoke({"language" : "영어", "question" : "주말에 할 일 알려줘. 콤마로 구분해서 줘"})
print(result.text)

Clean the house, do the laundry, go grocery shopping, exercise, meet friends or family, read a book, watch a movie, prepare for the upcoming week


### 연습하기
1. 영화 취향 입력받아 영화 3개 추천 - 콤마 리스트
2. 사용자 기분에 맞춰서 음악 출력 - str
3. 캐릭터 설정해서 캐릭터의 톤, 말투에 맞춰서 답변하는 체인 만들기

In [17]:
#1

prompt = ChatPromptTemplate.from_messages([
    ("system", "이제부터 {user_taste}에 맞춤 영화 3편 추천해주되, 제목(연도) 포맷으로, 콤마로 구분해서 줘"),
    ("human", "나는 {user_taste} 이런 느낌의 영화가 좋더라")
])

chain = prompt | model | CommaSeparatedListOutputParser()
result = chain.invoke({"user_taste" : "홍콩 느와르"})
print(result)

['영웅본색(1986)', '첩혈쌍웅(1989)', '무간도(2002)']


In [22]:
#2

prompt = ChatPromptTemplate.from_messages([
    ("system", "이제부터 사용자의 {emotion}에 맞춤 음악 추천해 줘"),
    ("human", "내 지금 기분은... {emotion}.")
])

chain = prompt | model | StrOutputParser()
result = chain.invoke({"emotion" : "분위기를 원해"})
print(result)

지금은 설명하기보다 **분위기에 잠기고 싶은 기분** 같아.  
은은하고 몽환적인 곡들로 추천할게.

- **Men I Trust – Show Me How**: 나른하고 부드러운 밤
- **Cigarettes After Sex – Apocalypse**: 고요하고 쓸쓸한 로맨스
- **wave to earth – seasons**: 잔잔하게 감정이 번지는 느낌
- **DPR IAN – So Beautiful**: 세련되고 어두운 몽환감
- **백예린 – Square (2017)**: 혼자 있고 싶은 저녁
- **검정치마 – 기다린 만큼, 더**: 담담하면서 깊은 여운

한 곡만 고르면 **Men I Trust – Show Me How**부터 들어봐.


In [25]:
#3

prompt = ChatPromptTemplate.from_messages([
    ("system", "이제부터 사용자의 {character}에 맞게 대사 바꿔줘"),
    ("human", "{script}")
])

chain = prompt | model | StrOutputParser()
result = chain.invoke({"character" : "경상도 토박이", "script" : "너 지금 말 다했어?"})
print(result)

니 지금 말 다 했나?


### 구조화된 출력
- 예전에는 pydantic으로 강제로 했다면,
- 요즘엔 model에서 구조화된 출력을 지원함

In [ ]:
from pydantic import BaseModel, Field

class Place(BaseModel):
    name : str = Field(description="추천 여행 장소명")
    reason : str = Field(description="추천한 이유")
    date : str = Field(description="추천 여행일")

structured_model = model.with_structured_output(Place, method="json_schema")    # 이런 식으로 모델에서 구조화된 출력 자체를 지원할 때 있다!

response = structured_model.invoke("주말에 갈만한 여행지 추천해 줘.")
print(response)

name='강릉' reason='서울에서 KTX로 이동하기 편하고, 안목해변 카페거리·경포호·초당순두부 등 바다와 먹거리를 함께 즐길 수 있어 주말 여행에 좋습니다.' date='토요일~일요일 1박 2일'


In [32]:
# result = response.__dict__
result = response.model_dump()  # 이 함수로 딕셔너리로 바꿀 수 있음!
print(result)
type(result)

{'name': '강릉', 'reason': '서울에서 KTX로 이동하기 편하고, 안목해변 카페거리·경포호·초당순두부 등 바다와 먹거리를 함께 즐길 수 있어 주말 여행에 좋습니다.', 'date': '토요일~일요일 1박 2일'}


dict